# Before You Refresh

**A client-wide check for a page review queue**  
Finley Septhian Darma · FlyRank ML Internship · October 2026

## Abstract

When many pages for one client lose search visibility together, should an editor check the broader pattern before refreshing individual pages? I use the gated FlyRank warehouse release to compare May search signals with observed June declines and to examine complete May–June page windows by client. On the held-out clients, the simple rule reaches 95% precision among its top 20 pages, compared with 80% for a logistic model, against a 65.5% decline base rate. A separate end-of-June check flags 14 of 32 clients with enough matched pages under the stated broad-drop rule. These observations support a human investigation step before page-level edits, without identifying a decline's cause or measuring refresh effects.

This notebook contains the calculations behind the [public paper](https://finleydarma.me/flyrank-ml-internship/work/paper/). It prints aggregates only. Local page-level caches and DuckDB spill files remain under work/outputs and are excluded from Git.


## 1. Question

I want to know whether a client-level decline check changes where an editor should look before acting on a page review queue. The unit of the May prediction check is one content item for one client at the May 31 cutoff. The separate June diagnostic uses complete May–June pages to classify the scope of a decline within a client. A flag changes the order of human review; it does not remove pages or choose an edit.


## 2. Data and coverage

The source is the gated FlyRank Internship warehouse release v20260703. Its daily performance fact has 78,835,655 rows in the full release; this analysis reads only the May and June 2026 partitions. May 2–31 supplies the 30-day input window and June 1–30 supplies the observed outcome window. A historical page needs all 30 daily observations in each month, nonmissing impressions and clicks, and at least 100 May impressions. The June review queue separately requires all 30 June days and at least 100 June impressions.

The raw daily source has repeated keys in the June sample. The loader checks that repeated keys have the same GSC values, keeps one value per client, page, and day, and then aggregates to page windows. The local cache is used when present; a fresh clone rebuilds it from the gated release with a Hugging Face read token supplied through HF_TOKEN or a hidden prompt. No token is written to the notebook.

I use GSC impressions, clicks, and average position. I leave out the fixed 90-day query table because its recent windows overlap the June label, and leave out GA4 because early availability differs across clients. Hashed IDs are only join and split keys. No URLs, client names, page titles, or raw queries are loaded or printed.


In [1]:
import getpass
import os
import tempfile
from datetime import date, timedelta
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLUMNS = (
    'log_prior30_impressions', 'log_prior30_clicks', 'prior30_ctr',
    'prior30_avg_position_filled', 'position_missing',
)

LABEL_END = date(2026, 6, 30)
HISTORICAL_CUTOFF = LABEL_END - timedelta(days=30)
FEATURE_START = HISTORICAL_CUTOFF - timedelta(days=29)
CURRENT_START = HISTORICAL_CUTOFF + timedelta(days=1)

def repo_root() -> Path:
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'work' / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('Open the repository or a folder inside it before running the notebook')

def _quote_path(path: Path) -> str:
    return str(path).replace('\\', '/').replace("'", "''")

def _feature_select(table: str, labeled: bool) -> str:
    extra = (
        ", future30_impressions, CASE WHEN future30_impressions < 0.8 * prior30_impressions "
        "THEN 1 ELSE 0 END AS future_decline_label"
    ) if labeled else ''
    return f"""
        SELECT client_hash_id, content_hash_id, prior30_impressions, prior30_clicks,
               prior30_avg_position,
               LN(1 + prior30_impressions) AS log_prior30_impressions,
               LN(1 + prior30_clicks) AS log_prior30_clicks,
               CAST(prior30_clicks AS DOUBLE) / prior30_impressions AS prior30_ctr,
               COALESCE(prior30_avg_position, 0.0) AS prior30_avg_position_filled,
               CASE WHEN prior30_avg_position IS NULL THEN 1 ELSE 0 END AS position_missing
               {extra}
        FROM {table}
    """

repo = repo_root()


In [2]:
def _summarize_month(con: duckdb.DuckDBPyConnection, source: str,
                     start: date, end: date, folder: Path) -> Path:
    """Stream a month to disk, then deduplicate one day at a time."""
    folder.mkdir()
    stage = folder / 'daily_source.parquet'
    days = folder / 'days'
    days.mkdir()
    summary = folder / 'summary.parquet'
    con.execute(f"""
        COPY (SELECT report_date, client_hash_id, content_hash_id,
                     gsc_impressions, gsc_clicks, gsc_avg_position
              FROM {source}
              WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}')
        TO '{_quote_path(stage)}' (FORMAT PARQUET)
    """)
    day = start
    while day <= end:
        day_path = days / f'{day}.parquet'
        con.execute(f"""
            COPY (
                SELECT client_hash_id, content_hash_id,
                       MIN(gsc_impressions) AS impressions,
                       MIN(gsc_clicks) AS clicks,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_avg_position) * MIN(gsc_impressions)
                            ELSE 0 END AS position_numerator,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_impressions) ELSE 0 END AS position_denominator,
                       COUNT(*) - 1 AS repeated_rows,
                       CASE WHEN MIN(gsc_impressions) IS DISTINCT FROM MAX(gsc_impressions)
                                      OR MIN(gsc_clicks) IS DISTINCT FROM MAX(gsc_clicks)
                                      OR MIN(gsc_avg_position) IS DISTINCT FROM MAX(gsc_avg_position)
                                      OR (COUNT(gsc_impressions) > 0 AND COUNT(gsc_impressions) < COUNT(*))
                                      OR (COUNT(gsc_clicks) > 0 AND COUNT(gsc_clicks) < COUNT(*))
                                      OR (COUNT(gsc_avg_position) > 0 AND COUNT(gsc_avg_position) < COUNT(*))
                            THEN 1 ELSE 0 END AS conflicting_keys
                FROM read_parquet('{_quote_path(stage)}')
                WHERE report_date = DATE '{day}'
                GROUP BY client_hash_id, content_hash_id
            ) TO '{_quote_path(day_path)}' (FORMAT PARQUET)
        """)
        day += timedelta(days=1)
    stage.unlink()
    con.execute(f"""
        COPY (
            SELECT client_hash_id, content_hash_id, COUNT(*) AS observed_days,
                   COUNT(*) FILTER (WHERE impressions IS NULL OR clicks IS NULL) AS missing_days,
                   SUM(impressions) AS impressions, SUM(clicks) AS clicks,
                   SUM(position_numerator) / NULLIF(SUM(position_denominator), 0) AS avg_position,
                   SUM(repeated_rows) AS repeated_rows,
                   SUM(conflicting_keys) AS conflicting_keys
            FROM read_parquet('{_quote_path(days / '*.parquet')}')
            GROUP BY client_hash_id, content_hash_id
        ) TO '{_quote_path(summary)}' (FORMAT PARQUET)
    """)
    for day_path in days.glob('*.parquet'):
        day_path.unlink()
    days.rmdir()
    return summary


In [3]:
def build_frames(source_override: str | None = None,
                 connection: duckdb.DuckDBPyConnection | None = None
                 ) -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    output.mkdir(parents=True, exist_ok=True)
    con = connection or duckdb.connect()
    with tempfile.TemporaryDirectory(prefix='lane2_build_', dir=output) as scratch:
        temp = Path(scratch)
        spill = output / 'lane2_duckdb_spill'
        spill.mkdir(exist_ok=True)
        con.execute(f"SET temp_directory = '{_quote_path(spill)}'")
        con.execute("SET memory_limit = '1GB'")
        con.execute('SET threads = 2')
        if source_override is None:
            token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
            con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
                        + token.replace("'", "''") + "')")
            del token
            root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
            may_source = f"read_parquet('{root}/month=2026-05/*.parquet')"
            june_source = f"read_parquet('{root}/month=2026-06/*.parquet')"
        else:
            may_source = june_source = source_override

        may = _summarize_month(con, may_source, FEATURE_START, HISTORICAL_CUTOFF,
                               temp / 'may')
        june = _summarize_month(con, june_source, CURRENT_START, LABEL_END,
                                temp / 'june')
        may_rel = f"read_parquet('{_quote_path(may)}')"
        june_rel = f"read_parquet('{_quote_path(june)}')"
        counts = con.sql(f"""
            SELECT SUM(repeated_rows), SUM(conflicting_keys),
                   COUNT(*) FILTER (WHERE client_hash_id IS NULL OR content_hash_id IS NULL)
            FROM (SELECT * FROM {may_rel} UNION ALL SELECT * FROM {june_rel})
        """).fetchone()
        repeated, conflicting, missing_keys = (int(value or 0) for value in counts)
        assert missing_keys == 0, f'{missing_keys:,} rows have a missing daily key'
        assert conflicting == 0, f'{conflicting:,} daily keys have different GSC values'

        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE historical_metrics AS
            SELECT m.client_hash_id, m.content_hash_id,
                   m.impressions AS prior30_impressions,
                   m.clicks AS prior30_clicks,
                   m.avg_position AS prior30_avg_position,
                   j.impressions AS future30_impressions
            FROM {may_rel} m
            JOIN {june_rel} j USING (client_hash_id, content_hash_id)
            WHERE m.observed_days = 30 AND m.missing_days = 0
              AND m.impressions >= 100
              AND j.observed_days = 30 AND j.missing_days = 0
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE current_metrics AS
            SELECT client_hash_id, content_hash_id,
                   impressions AS prior30_impressions,
                   clicks AS prior30_clicks,
                   avg_position AS prior30_avg_position
            FROM {june_rel}
            WHERE observed_days = 30 AND missing_days = 0 AND impressions >= 100
        """)
        historical = con.sql(_feature_select('historical_metrics', labeled=True)).df()
        current = con.sql(_feature_select('current_metrics', labeled=False)).df()

    assert len(historical) > 0 and len(current) > 0, 'No pages met the coverage and impression rules'
    assert historical[list(FEATURE_COLUMNS)].notna().all().all()
    assert current[list(FEATURE_COLUMNS)].notna().all().all()
    print(f'May-June repeated GSC rows removed: {repeated:,}')
    print(f'Historical labeled pages: {len(historical):,}; current June pages: {len(current):,}')
    return historical, current

def load_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    history_path = output / 'lane2_history_2026-05-31.parquet'
    current_path = output / 'lane2_current_2026-06-30.parquet'
    if history_path.exists() and current_path.exists():
        con = duckdb.connect()
        history = con.sql(f"SELECT * FROM read_parquet('{_quote_path(history_path)}')").df()
        current = con.sql(f"SELECT * FROM read_parquet('{_quote_path(current_path)}')").df()
        print(f'Loaded cached frames: {len(history):,} historical, {len(current):,} current pages')
        return history, current
    history, current = build_frames()
    output.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.register('cache_history', history)
    con.register('cache_current', current)
    con.execute(f"COPY cache_history TO '{_quote_path(history_path)}' (FORMAT PARQUET)")
    con.execute(f"COPY cache_current TO '{_quote_path(current_path)}' (FORMAT PARQUET)")
    print('Saved local Parquet caches under work/outputs; dataset files are gitignored')
    return history, current


In [4]:
history, current = load_frames()
required_history = {'client_hash_id', 'content_hash_id', 'prior30_impressions',
                    'future30_impressions', 'future_decline_label', *FEATURE_COLUMNS}
assert required_history.issubset(history.columns)
assert 'future30_impressions' not in current.columns
assert 'future_decline_label' not in current.columns
assert not history.duplicated(['client_hash_id', 'content_hash_id']).any()
assert not current.duplicated(['client_hash_id', 'content_hash_id']).any()
assert (history.prior30_impressions >= 100).all()
assert (current.prior30_impressions >= 100).all()
assert (history.future_decline_label.astype(int) ==
        (history.future30_impressions < 0.8 * history.prior30_impressions).astype(int)).all()
print(f'Historical complete-window pages: {len(history):,} across {history.client_hash_id.nunique()} clients')
print(f'June review-queue pages: {len(current):,} across {current.client_hash_id.nunique()} clients')
print('Historical label: June impressions more than 20% below May impressions')


Loaded cached frames: 102,532 historical, 96,678 current pages
Historical complete-window pages: 102,532 across 42 clients
June review-queue pages: 96,678 across 44 clients
Historical label: June impressions more than 20% below May impressions


## 3. Methodology

The five model inputs are the logarithms of May impressions and clicks, May click-through rate, May average position with a separate missing-position indicator, and no other fields. Zero is a computation placeholder for missing position, never interpreted as a real position because the indicator stays with it. The label is 1 when June impressions are more than 20% below May impressions. It records a later search decline, not the effect of a refresh.

The baseline is the Week 5–7 transparent review score: prioritize pages above the training set's median May impressions, add a point for low May CTR, add a point for May average position from 11 through 20, and use a small impression term to order ties. The low-CTR cutoff is the training set's lower quartile. Both rule cutoffs come from training clients only. The comparison model is scaled logistic regression using the same five May inputs.

One GroupShuffleSplit holds out 25% of clients (seed 42). The rule and model use the same held-out pages and observed June labels. This tests unseen clients at one May cutoff, not a later time period. June columns and client IDs are absent from the model input list.


In [5]:
def grouped_split(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    groups = frame['client_hash_id']
    if groups.nunique() < 2:
        raise ValueError('A grouped split needs at least two clients')
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=42).split(frame, groups=groups))
    assert set(groups.iloc[train]).isdisjoint(set(groups.iloc[test]))
    return train, test

def baseline_settings(frame: pd.DataFrame, train: np.ndarray) -> tuple[float, float]:
    part = frame.iloc[train]
    return float(part['prior30_impressions'].median()), float(part['prior30_ctr'].quantile(0.25))

def baseline_rank(frame: pd.DataFrame, settings: tuple[float, float]) -> pd.DataFrame:
    visibility_min, low_ctr_max = settings
    visible = frame['prior30_impressions'] >= visibility_min
    low_ctr = frame['prior30_ctr'] <= low_ctr_max
    striking = frame['prior30_avg_position'].between(11, 20, inclusive='both')
    result = frame[['client_hash_id', 'content_hash_id', 'prior30_impressions',
                    'prior30_clicks', 'prior30_ctr', 'prior30_avg_position', 'position_missing']].copy()
    result['baseline_score'] = (visible.astype(int) * (2 + low_ctr.astype(int) + striking.astype(int))
                                + 0.01 * np.log1p(frame['prior30_impressions']))
    result['reason_code'] = np.select(
        [visible & low_ctr & striking, visible & low_ctr, visible & striking, visible],
        ['visible_low_ctr_striking', 'visible_low_ctr', 'visible_striking', 'visible_page'],
        default='low_visibility_review_later',
    )
    return result

def precision_at_k(scores, labels, k: int) -> tuple[float, int]:
    values = np.asarray(scores, dtype=float)
    truth = np.asarray(labels, dtype=int)
    if len(values) != len(truth) or len(values) == 0:
        raise ValueError('Scores and labels need the same nonzero length')
    used = min(k, len(values))
    order = np.argsort(-values, kind='stable')[:used]
    return float(truth[order].mean()), used


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = list(FEATURE_COLUMNS)
assert len(features) == 5
assert not any('future' in name or 'client' in name or 'content' in name for name in features)
assert history[features].notna().all().all()
y = history.future_decline_label.astype(int)
train_idx, test_idx = grouped_split(history)
train_clients = set(history.client_hash_id.iloc[train_idx])
test_clients = set(history.client_hash_id.iloc[test_idx])
assert train_clients.isdisjoint(test_clients)
settings = baseline_settings(history, train_idx)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
model.fit(history[features].iloc[train_idx], y.iloc[train_idx])
model_scores = model.predict_proba(history[features].iloc[test_idx])[:, 1]
rule_scores = baseline_rank(history, settings).baseline_score.iloc[test_idx]
test_labels = y.iloc[test_idx]
precision = {}
for k in (20, 50):
    rule_p, used = precision_at_k(rule_scores, test_labels, k)
    model_p, _ = precision_at_k(model_scores, test_labels, k)
    precision[k] = {'rule': rule_p, 'model': model_p}
majority_class = int(y.iloc[train_idx].mean() >= 0.5)
majority_accuracy = accuracy_score(test_labels, np.full(len(test_idx), majority_class))
model_accuracy = accuracy_score(test_labels, model_scores >= 0.5)
base_rate = float(test_labels.mean())
print(f'Held-out clients: {len(test_clients)}; held-out pages: {len(test_idx):,}')
print(f'Observed June decline base rate: {base_rate:.1%}')
for k in (20, 50):
    print(f'Precision at {k}: rule {precision[k]["rule"]:.1%}; model {precision[k]["model"]:.1%}')
print(f'Majority accuracy: {majority_accuracy:.1%}; model accuracy: {model_accuracy:.1%}')
print('The simple rule stays in the review queue on this split.')


Held-out clients: 11; held-out pages: 10,763
Observed June decline base rate: 65.5%
Precision at 20: rule 95.0%; model 80.0%
Precision at 50: rule 92.0%; model 82.0%
Majority accuracy: 65.5%; model accuracy: 63.7%
The simple rule stays in the review queue on this split.


## 4. Results: broad decline as a second review step

For the end-of-June descriptive check, I use historical pages with complete May and June windows and at least 100 **May** impressions. I do not require 100 June impressions: that would discard pages that fell below the June queue threshold and undercount severe declines. A page has a decline when June impressions are more than 20% below May. A client is flagged only when at least 30 such matched pages are available, more than 60% of them decline, and their combined June impressions fall by more than 20% from May.

This June check is computed after the May prediction test. It is never fed into the May model or rule. A client with too few matched pages gets a coverage-check route rather than a negative broad-drop result.


In [7]:
matched = history[['client_hash_id', 'content_hash_id', 'prior30_impressions',
                   'future30_impressions']].copy()
matched['page_declined'] = matched.future30_impressions < 0.8 * matched.prior30_impressions
client_summary = matched.groupby('client_hash_id').agg(
    matched_pages=('content_hash_id', 'size'),
    declining_pages=('page_declined', 'sum'),
    may_impressions=('prior30_impressions', 'sum'),
    june_impressions=('future30_impressions', 'sum'),
)
client_summary['declining_share'] = client_summary.declining_pages / client_summary.matched_pages
client_summary['combined_drop'] = 1 - client_summary.june_impressions / client_summary.may_impressions
client_summary['enough_pages'] = client_summary.matched_pages >= 30
client_summary['broad_flag'] = (client_summary.enough_pages
    & (client_summary.declining_share > 0.60)
    & (client_summary.combined_drop > 0.20))
enough_count = int(client_summary.enough_pages.sum())
broad_count = int(client_summary.broad_flag.sum())
print(f'Matched complete-window pages: {len(matched):,} across {len(client_summary)} clients')
print(f'Clients with at least 30 matched pages: {enough_count}')
print(f'Broad-drop flags under the main rule: {broad_count}')
print(f'Clients below the 30-page minimum: {int((~client_summary.enough_pages).sum())}')


Matched complete-window pages: 102,532 across 42 clients
Clients with at least 30 matched pages: 32
Broad-drop flags under the main rule: 14
Clients below the 30-page minimum: 10


In [8]:
def count_flags(min_pages=30, page_loss=0.20, share=0.60, combined_loss=0.20):
    falling = matched.future30_impressions < (1 - page_loss) * matched.prior30_impressions
    fraction = falling.groupby(matched.client_hash_id).mean()
    return int(((client_summary.matched_pages >= min_pages)
                & (fraction > share)
                & (client_summary.combined_drop > combined_loss)).sum())

for value in (0.10, 0.20, 0.30):
    print(f'Page-loss cutoff >{value:.0%}: {count_flags(page_loss=value)} flagged clients')
for value in (0.50, 0.60, 0.70):
    print(f'Declining-page share >{value:.0%}: {count_flags(share=value)} flagged clients')
for value in (0.10, 0.20, 0.30):
    print(f'Combined loss >{value:.0%}: {count_flags(combined_loss=value)} flagged clients')
for value in (20, 30, 40):
    print(f'Minimum matched pages {value}: {count_flags(min_pages=value)} flagged clients')
assert count_flags() == broad_count


Page-loss cutoff >10%: 14 flagged clients
Page-loss cutoff >20%: 14 flagged clients
Page-loss cutoff >30%: 11 flagged clients
Declining-page share >50%: 14 flagged clients
Declining-page share >60%: 14 flagged clients
Declining-page share >70%: 11 flagged clients
Combined loss >10%: 15 flagged clients
Combined loss >20%: 14 flagged clients
Combined loss >30%: 10 flagged clients
Minimum matched pages 20: 15 flagged clients
Minimum matched pages 30: 14 flagged clients
Minimum matched pages 40: 14 flagged clients


In [9]:
queue = baseline_rank(current, settings).sort_values('baseline_score', ascending=False, kind='stable').copy()
has_broad_flag = queue.client_hash_id.map(client_summary.broad_flag).eq(True)
has_enough_pages = queue.client_hash_id.map(client_summary.enough_pages).eq(True)
queue['review_route'] = np.select(
    [has_broad_flag, has_enough_pages],
    ['Investigate broader pattern first', 'Review page evidence'],
    default='Check history coverage first',
)
route_counts = queue.review_route.value_counts()
for route in ('Investigate broader pattern first', 'Review page evidence',
              'Check history coverage first'):
    print(f'{route}: {int(route_counts.get(route, 0)):,} June queue pages')
top20_broad = int((queue.head(20).review_route == 'Investigate broader pattern first').sum())
print(f'Top-20 rule-ranked June pages in the broad route: {top20_broad}')
assert int(route_counts.sum()) == len(current)
print('Every page remains in the queue; each route requires a human decision.')


Investigate broader pattern first: 66,716 June queue pages
Review page evidence: 27,938 June queue pages
Check history coverage first: 2,024 June queue pages
Top-20 rule-ranked June pages in the broad route: 11
Every page remains in the queue; each route requires a human decision.


## 5. Limitations

This is one cutoff and one client-held-out split, so the precision numbers are directional. The high decline base rate makes top-k precision easier to achieve. The broad-drop thresholds are review heuristics, not learned or validated cutoffs. The complete-window and May-impression rules exclude newer or smaller pages, and large clients contribute many more pages than small clients. A client-wide signal does not tell us whether demand, indexing, a shared template, or another factor caused the change. Search Console reporting has privacy and row limits. July outcomes are absent, and no refresh intervention was measured.


In [10]:
assert 'future30_impressions' not in features
assert 'future_decline_label' not in features
assert train_clients.isdisjoint(test_clients)
assert not current.columns.str.contains('future|label', case=False).any()
print('Leakage checks passed: May-only model inputs, disjoint clients, no July claim.')


Leakage checks passed: May-only model inputs, disjoint clients, no July claim.


## 6. Ranked recommendations

1. Check history coverage first. If a client has fewer than 30 matched complete-window pages, gather more evidence before calling the decline broad or local.
2. For a flagged client, inspect the shared pattern before editing pages: check Search Console coverage and indexing, compare clicks and positions, review seasonality or demand, and check any common site or template changes. The flag supplies a scope question, not a cause.
3. For clients without a broad flag, use the transparent page score to choose pages for human review. Inspect accuracy, search intent, titles, snippets, relevance, and internal links before deciding whether a refresh, CTR edit, or no change is appropriate.
4. Record the decision and monitor a later window. Only a future evaluation can show whether the routing rule generalizes; measuring refresh impact needs an intervention design.


## 7. Figures for the public paper

These SVGs contain aggregate counts and metrics only. Their captions and values also appear as text in the paper.


In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

figure_dir = repo / 'work' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
ink, teal, rust, sage, paper = '#18342f', '#285c58', '#b25332', '#abc5b8', '#fbfaf6'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.edgecolor': ink,
                     'axes.labelcolor': ink, 'text.color': ink, 'xtick.color': ink,
                     'ytick.color': ink, 'svg.fonttype': 'none'})

fig, ax = plt.subplots(figsize=(8.8, 4.5), facecolor=paper)
ax.set_facecolor(paper)
x = np.arange(2)
width = 0.31
rule_values = [precision[k]['rule'] * 100 for k in (20, 50)]
model_values = [precision[k]['model'] * 100 for k in (20, 50)]
bars_rule = ax.bar(x - width/2, rule_values, width, label='Simple rule', color=teal)
bars_model = ax.bar(x + width/2, model_values, width, label='Logistic model', color=rust, hatch='//')
ax.axhline(base_rate * 100, color=ink, linestyle='--', linewidth=1.3,
           label=f'Decline base rate ({base_rate:.1%})')
for bar in [*bars_rule, *bars_model]:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{bar.get_height():.0f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_xticks(x, ['Top 20 pages', 'Top 50 pages'])
ax.set_ylabel('Observed June declines among selected pages (%)')
ax.set_ylim(0, 112)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.16), ncol=3, frameon=False, fontsize=10)
fig.tight_layout()
fig.savefig(figure_dir / 'capstone_model_comparison.svg', bbox_inches='tight',
            metadata={'Date': None})
plt.close(fig)

categories = ['Broad flag', 'No broad flag', 'Too few matched pages']
counts = [broad_count, enough_count - broad_count, len(client_summary) - enough_count]
fig, ax = plt.subplots(figsize=(8.8, 3.9), facecolor=paper)
ax.set_facecolor(paper)
bars = ax.barh(categories[::-1], counts[::-1], color=[sage, '#859c91', rust])
for bar in bars:
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontweight='bold')
ax.set_xlim(0, max(counts) + 5)
ax.set_xlabel('Clients in the complete-window May–June subset')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(figure_dir / 'capstone_broad_drop.svg', bbox_inches='tight',
            metadata={'Date': None})
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(10.4, 3.9), facecolor=paper, sharey=True)
panels = [
    ('Page loss', ['10%', '20%', '30%'], [count_flags(page_loss=v) for v in (.1, .2, .3)]),
    ('Page share', ['50%', '60%', '70%'], [count_flags(share=v) for v in (.5, .6, .7)]),
    ('Combined loss', ['10%', '20%', '30%'], [count_flags(combined_loss=v) for v in (.1, .2, .3)]),
]
for ax, (title, labels, values) in zip(axes, panels):
    ax.set_facecolor(paper)
    bars = ax.bar(labels, values, color=[sage, teal, sage], width=0.62)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 19)
    ax.spines[['top', 'right']].set_visible(False)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.45,
                str(int(bar.get_height())), ha='center', fontweight='bold')
axes[0].set_ylabel('Flagged clients')
fig.tight_layout(w_pad=2.4)
fig.savefig(figure_dir / 'capstone_threshold_sensitivity.svg', bbox_inches='tight',
            metadata={'Date': None})
plt.close(fig)
for svg in figure_dir.glob('capstone_*.svg'):
    svg.write_text('\n'.join(line.rstrip() for line in svg.read_text(encoding='utf-8').splitlines()) + '\n',
                   encoding='utf-8')
print('Saved three aggregate SVG figures under work/figures.')


Saved three aggregate SVG figures under work/figures.


## 8. Reproducibility

Run from the repository root after installing requirements.txt and ipykernel. Open this notebook in VS Code and use Run All. A fresh clone needs access to the gated [FlyRank Internship warehouse release](https://huggingface.co/datasets/FlyRank/internship-warehouse) and a Hugging Face read token through HF_TOKEN or the hidden prompt. The first run streams only May and June partitions through DuckDB with a 1 GB memory limit and two threads, and saves local ignored caches under work/outputs. Later runs use those caches. The split seed and logistic regression seed are both 42. For the earlier data checks and queue work, see [Week 3](w03_feature_leakage_check.ipynb), [Week 5](w05_model.ipynb), [Week 6](w06_validation_audit.ipynb), and [Week 7](w07_action_playbook.ipynb).

The public repository contains code, aggregate figures, and the paper. It does not contain the gated page-level data.


## 9. Acknowledgments and data credit

[Built on the FlyRank ML Internship dataset](https://flyrank.ai). The gated release is credited above and in the public paper. Google Search Central's [traffic-drop debugging guide](https://developers.google.com/search/docs/monitor-debug/debugging-search-traffic-drops) informed the review question; it does not endorse these thresholds.
